In [7]:
import os
from pathlib import Path
# import cv2
import numpy as np
from tqdm import tqdm
# import mediapipe as mp

# mp_holistic = mp.solutions.holistic

BASE = Path("C:\\Users\\mishb\\OneDrive\\Desktop\\text2sign\\text2sign\\new\\WLASL")
VIDEO_DIR  = Path("C:\\Users\\mishb\\OneDrive\\Desktop\\text2sign\\text2sign\\new\\WLASL\\wlasl_1000_preproc\\videos")
TEST_NPY = Path("C:\\Users\\mishb\\OneDrive\\Desktop\\text2sign\\text2sign\\new\\WLASL\\wlasl_1000_preproc\\videos/1/01610.npy")


In [8]:
paths = {
    "BASE": BASE,
    "VIDEO_DIR": VIDEO_DIR,
    "TEST_NPY": TEST_NPY,
}

for name, path in paths.items():
    if path.exists():
        print(f"{name} exists at {path}")
    else:
        print(f"{name} does NOT exist at {path}")

BASE exists at C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL
VIDEO_DIR exists at C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos
TEST_NPY exists at C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\1\01610.npy


In [3]:
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class LSMU:
    '''
    One LSMU = one 'building block' of a sign.
    A single word sign may need 1 or 2 LSMUs.
    '''
    handshape:   str   = 'flat-b'      # e.g. 'W', 'flat-b', 'index-1', 'open-5'
    orientation: str   = 'palm-up'     # e.g. 'palm-up', 'palm-down', 'palm-left'
    movement:    str   = 'hold'        # e.g. 'tap', 'arc', 'circular', 'hold', 'shake'
    location:    str   = 'neutral'     # e.g. 'chin', 'temple', 'chest', 'neutral'
    # nms:         List[str] = field(default_factory=list)   # ['mouth-open', 'browraise']
    confidence:  float = 1.0           # 0.0 to 1.0
    flags:       Dict[str, bool] = field(default_factory=lambda: {
                     'plural': False,
                     'negation': False,
                     'intense': False,
                 })
    duration_ms: float = 300.0         # how long this LSMU lasts (filled by planner)

In [4]:
def split_landmarks(data):
    """
    Input: (T, 75, 3)
    Output: pose, left_hand, right_hand
    """
    pose = data[:, :33]
    left = data[:, 33:54]
    right = data[:, 54:]
    return pose, left, right

def select_dominant_hand(left, right):
    """
    Choose the more active hand
    """
    left_energy = np.mean(np.abs(left))
    right_energy = np.mean(np.abs(right))
    return left if left_energy > right_energy else right

def get_location(hand_seq, pose_seq):
    """
    hand_seq: (T,21,3)
    pose_seq: (T,33,3)
    """

    # average over time + landmarks → (3,)
    hand_center = np.mean(hand_seq, axis=(0,1))
    pose_avg = np.mean(pose_seq, axis=0)  # (33,3)

    nose = pose_avg[0]
    shoulder_center = (pose_avg[11] + pose_avg[12]) / 2

    # if hand_center[1] < nose[1]:
    #     return "head"
    # elif hand_center[1] < shoulder_center[1]:
    #     return "upper_body"
    # else:
    #     return "lower_body"
    
    relative_y = hand_center[1] - shoulder_center[1]

    if relative_y < -0.1:
        return "head"
    elif relative_y < 0.1:
        return "upper_body"
    else:
        return "lower_body"
    
def classify_handshape(hand):
    """
    Simple rule-based handshape classification
    """
    fingers = [
        (8, 6),   # index
        (12, 10), # middle
        (16, 14), # ring
        (20, 18)  # pinky
    ]

    states = []
    for tip, pip in fingers:
        dist = np.linalg.norm(hand[tip] - hand[pip])
        states.append(dist > 0.04)

    if all(states):
        return "open_palm"
    elif not any(states):
        return "fist"
    elif states[0] and not states[1]:
        return "pointing"
    else:
        return "partial_open"
    
def classify_movement(hand_seq):
    centers = np.mean(hand_seq, axis=1)  # (T,3)

    velocity = np.diff(centers, axis=0)  # (T-1,3)
    speed = np.linalg.norm(velocity, axis=1)

    total_motion = np.sum(speed)

    if total_motion < 0.5:
        return "static"

    dx = np.mean(velocity[:,0])
    dy = np.mean(velocity[:,1])

    if abs(dx) > abs(dy):
        return "horizontal"
    else:
        return "vertical"
    
def classify_orientation(hand):
    """
    Approximate palm orientation
    """
    v1 = hand[5] - hand[0]    # index base
    v2 = hand[17] - hand[0]   # pinky base

    normal = np.cross(v1, v2)

    if normal[2] > 0:
        return "palm_forward"
    else:
        return "palm_backward"
    

In [5]:
def process_npy(file_path):
    data = np.load(file_path)  # (T,75,3)

    pose, left, right = split_landmarks(data)
    hand_seq = select_dominant_hand(left, right)

    avg_hand = np.mean(hand_seq, axis=0)

    location = get_location(hand_seq, pose)
    handshape = classify_handshape(avg_hand)
    movement = classify_movement(hand_seq)
    orientation = classify_orientation(avg_hand)

    return {
        "location": location,
        "handshape": handshape,
        "movement": movement,
        "orientation": orientation
    }

In [9]:
res = process_npy(TEST_NPY)

In [32]:
print(res)

{'location': 'head', 'handshape': 'fist', 'movement': 'horizontal', 'orientation': 'palm_backward'}


In [14]:
# ──────────────────────────────────────────────
# Batch runner
# ──────────────────────────────────────────────
import json
import argparse
def batch_process(root_dir: str):
    """
    Walk every sub-folder under root_dir.
    For each  <ID>.npy  file found, write  <ID>.json  in the same folder.
    """
    processed = 0
    skipped   = 0
    errors    = 0
 
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if not fname.lower().endswith(".npy"):
                continue
 
            npy_path  = os.path.join(dirpath, fname)
            file_id   = os.path.splitext(fname)[0]          # e.g. "01610"
            json_path = os.path.join(dirpath, f"{file_id}.json")
 
            # Skip if JSON already exists (remove this check to re-process)
            if os.path.exists(json_path):
                skipped += 1
                continue
 
            try:
                features = process_npy(npy_path)
                record   = {"id": file_id, **features}      # id first, then features
 
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(record, f, indent=2)
 
                processed += 1
                print(f"[OK]  {json_path}")
 
            except Exception as exc:
                errors += 1
                print(f"[ERR] {npy_path}  →  {exc}")
 
    print(f"\nDone.  processed={processed}  skipped={skipped}  errors={errors}")
 

In [12]:
features = process_npy(TEST_NPY)
record = {"id": "01610", **features}

with open("test.json", "w", encoding="utf-8") as f:
    json.dump(record, f, indent=2)

In [16]:
import os
import json
import numpy as np

ROOT = "C:\\Users\\mishb\\OneDrive\\Desktop\\text2sign\\text2sign\\new\\WLASL\\wlasl_1000_preproc\\videos"


# ──────────────────────────────────────────────
# Batch runner
# ──────────────────────────────────────────────
 
def batch_process(root_dir: str):
    """
    Walk every sub-folder under root_dir.
    For each  <ID>.npy  file found, write  <ID>.json  in the same folder.
    """
    processed = 0
    skipped   = 0
    errors    = 0
 
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if not fname.lower().endswith(".npy"):
                continue
 
            npy_path  = os.path.join(dirpath, fname)
            file_id   = os.path.splitext(fname)[0]          # e.g. "01610"
            json_path = os.path.join(dirpath, f"{file_id}.json")
 
            # Skip if JSON already exists (remove this check to re-process)
            if os.path.exists(json_path):
                skipped += 1
                continue
 
            try:
                features = process_npy(npy_path)
                record   = {"id": file_id, **features}      # id first, then features
 
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(record, f, indent=2)
 
                processed += 1
                print(f"[OK]  {json_path}")
 
            except Exception as exc:
                errors += 1
                print(f"[ERR] {npy_path}  →  {exc}")
 
    print(f"\nDone.  processed={processed}  skipped={skipped}  errors={errors}")
 
 
# ──────────────────────────────────────────────
# Run — just execute this cell in your notebook
# ──────────────────────────────────────────────
 
print(f"Scanning: {ROOT}\n")
batch_process(ROOT)

Scanning: C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos

[OK]  C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\1\01610.json
[OK]  C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\1\01612.json
[OK]  C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\1\01615.json
[OK]  C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\1\66039.json
[OK]  C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\10\00663.json
[OK]  C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\10\00664.json
[OK]  C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\10\00666.json
[OK]  C:\Users\mishb\OneDrive\Desktop\text2sign\text2sign\new\WLASL\wlasl_1000_preproc\videos\10\00668.json
[OK]  C:\Users\mishb\OneDrive\Desktop\tex